In [19]:
# A. Membaca data langsung dari HDFS ke variabel df
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, 
    inferSchema=True
)

# Menampilkan skema, jumlah baris, dan 10 baris pertama
df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [20]:
# B. Menangani Data Kosong 
from pyspark.sql.functions import col

# Menghitung jumlah baris yang memiliki nilai kosong pada kolom rating
jumlah_null = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris kosong (NULL) pada kolom rating: {jumlah_null}")

# Menangani nilai null dengan mengisi angka 0 menggunakan format dictionary
df = df.na.fill({"rating": 0})

# Memastikan bahwa nilai null sudah tidak ada lagi
jumlah_null2 = df.filter(col("rating").isNull()).count()
print(f"Jumlah null pada kolom rating setelah ditangani: {jumlah_null2}")

Jumlah baris kosong (NULL) pada kolom rating: 204
Jumlah null pada kolom rating setelah ditangani: 0


Kita memilih df.na.fill({"rating": 0}) daripada na.drop() agar catatan pendapatan dari transaksi tersebut tidak ikut terhapus. Dengan mengisinya angka 0, data total pendapatan tetap utuh untuk dihitung, sekaligus menjadi penanda bahwa pembeli tidak memberikan rating.

In [21]:
# C. Transformasi Data
from pyspark.sql.functions import when

# Membuat kolom total_pendapatan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# Membuat kolom tier_transaksi berdasarkan kondisi
df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df.select("order_id", "total_pendapatan", "tier_transaksi").show(5)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows



In [22]:
# D. Analisis dengan GroupBy
from pyspark.sql.functions import sum as spark_sum, count, avg

# 1. Kategori dengan total_pendapatan tertinggi
print("1. Kategori Pendapatan Tertinggi:")
df.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total")) \
    .orderBy(col("total").desc()).show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("2. Kota dengan Transaksi Tier 'Besar' Terbanyak:")
df.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota").count() \
    .orderBy(col("count").desc()).show(1)

# 3. Rata-rata rating per metode_pembayaran
print("3. Rata-rata Rating per Metode Pembayaran:")
df.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")).show()

1. Kategori Pendapatan Tertinggi:
+------------+---------+
|    kategori|    total|
+------------+---------+
|Rumah Tangga|138665000|
+------------+---------+
only showing top 1 row

2. Kota dengan Transaksi Tier 'Besar' Terbanyak:
+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

3. Rata-rata Rating per Metode Pembayaran:
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [23]:
# E. Menyimpan Hasil ke HDFS dan Verifikasi
path_simpan_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transformasi_csv"

# Menyimpan DataFrame ke HDFS dalam format CSV
df.write.mode("overwrite").csv(path_simpan_hdfs, header=True)
print("Berhasil menyimpan data ke HDFS!")

# Verifikasi: Membaca kembali data yang baru disimpan dari HDFS
df_verifikasi = spark.read.csv(path_simpan_hdfs, header=True, inferSchema=True)
print("Verifikasi Berhasil! Jumlah baris data yang dibaca kembali:", df_verifikasi.count())
df_verifikasi.show(5)

Berhasil menyimpan data ke HDFS!
Verifikasi Berhasil! Jumlah baris data yang dibaca kembali: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00

Spark menyimpan hasil akhir sebagai beberapa berkas partisi alih-alih satu berkas tunggal karena arsitektur Spark dirancang untuk pemrosesan terdistribusi. Spark memecah data menjadi potongan-potongan kecil (partisi) yang diproses dan disimpan secara paralel oleh berbagai mesin (executor) secara bersamaan. Hal ini membuat proses penulisan dan pembacaan data berukuran raksasa menjadi jauh lebih cepat dan efisien.